# Synthetic (SMT) sweep analysis

Analyses the synthetic sweep (`cluster/`): a multi-head MLP trained on datasets of
increasing difficulty, with and without the learned-abstention head. The focus is the
abstention story -- why harder datasets are harder, and how much abstaining buys back.

Paper-ready figures are written to `figs/fig_*.{png,pdf}` as they render:

| figure | what it shows |
|---|---|
| `fig_posterior_overlap` | why the hardest problem is harder: class posteriors / features overlap |
| `fig_risk_coverage` | risk vs coverage: abstention's coverage/accuracy trade-off across `o` |
| `fig_metrics_vs_o` | accuracy + macro-F1 across the payoff `o` sweep |
| `fig_hardness_hist` | harder tasks -> lower accuracy AND bigger abstention gain (two histograms) |
| `fig_selective_risk` | selective-risk curves per dataset (easy -> hard) vs confidence / CE / Bayes |

Prereq: run the sweep first (`bash cluster/submit.sh` on SLURM, or
`bash cluster/run_local.sh` locally), then run this notebook from the repo root.


## Load every run

Loads the manifest + each run's metrics into one dataframe, with helpers to slice by study and average over seeds.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd

# find the repo root by walking up until cluster/manifest.json appears
root = Path.cwd()
while not (root / "cluster" / "manifest.json").exists() and root != root.parent:
    root = root.parent
manifest = json.loads((root / "cluster" / "manifest.json").read_text())
print("runs in matrix:", len(manifest["runs"]))


def load_rows():
    """One flat row per run that produced a metrics file. Hyperparameters + study
    tags come from the manifest; scores from the run's metrics JSON."""
    rows, missing = [], []
    for r in manifest["runs"]:
        p = root / r["metrics"]
        if not p.exists():
            missing.append(r["run_id"]); continue
        m = json.loads(p.read_text())
        d, mech = m["defect_head"], m["mechanism_accuracy"]
        row = {
            "run_id": r["run_id"], "dataset": r["dataset"], "loss": r["loss"],
            "o": r["o"], "seed": r["seed"], "hidden": r.get("hidden"),
            "dropout": r.get("dropout"), "class_weight": r.get("class_weight"),
            "lr": r.get("lr"), "epochs": r.get("epochs"),
            "studies": r.get("studies", []),
            "defect_acc": d["accuracy"], "defect_wf1": d["weighted_f1"],
            "defect_macrof1": d["macro_f1"],
            "bayes": m["bayes_optimal_accuracy"],
            "gap_to_bayes": m["bayes_optimal_accuracy"] - d["accuracy"],
            "mech_joint": mech["joint_accuracy"],
            "risk_mae": m["risk_mae"],
        }
        ab = m.get("abstention")
        if ab:
            row["mean_abstain"] = ab["mean_abstain_prob"]
            half = ab["by_threshold"].get("0.5", {})
            row["coverage@0.5"] = half.get("coverage")
            row["selective_acc@0.5"] = half.get("selective_accuracy")
        rows.append(row)
    if missing:
        print(f"WARNING: {len(missing)} runs have no metrics yet (still training?):")
        print("  " + ", ".join(missing[:12]) + (" ..." if len(missing) > 12 else ""))
    return pd.DataFrame(rows)


df = load_rows()
print(f"loaded {len(df)} of {len(manifest['runs'])} runs")


def study(name):
    """Rows tagged with a study (core, payoff, capacity, dropout, lr, class_weight)."""
    if not len(df) or "studies" not in df.columns:
        return df.iloc[0:0]
    return df[df["studies"].apply(lambda s: name in (s or []))]


def seed_mean(frame, keys, metrics):
    """Average over seeds -> one row per config (keeps NaN-keyed groups, e.g. o=None)."""
    metrics = [m for m in metrics if m in frame.columns]
    return frame.groupby(keys, dropna=False)[metrics].mean().reset_index()


def core_grid():
    """(core rows, dataset order, losses, bar x-positions, bar width)."""
    core = study("core")
    datasets = list(manifest["datasets"])
    losses = [l for l in ["cascade", "abstention"] if l in set(core["loss"])] if len(core) else []
    x = np.arange(len(datasets)); w = 0.8 / max(len(losses), 1)
    return core, datasets, losses, x, w

## Plot style

Shared paper style + a `save_fig` helper that writes `figs/fig_*.{png,pdf}`.

In [ ]:
# Shared paper-quality plotting style + a figs/ saver. Consistent colors for the two
# losses everywhere; horizontal layouts + zoomed axes so small differences are legible.
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 220, "savefig.bbox": "tight",
    "font.size": 12, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.labelsize": 12, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False,
})
COLORS = {"cascade": "#2c7fb8", "abstention": "#e6550d", "bayes": "#333333"}

FIGDIR = root / "figs"
FIGDIR.mkdir(exist_ok=True)


def save_fig(fig, name):
    """Write a paper copy (PNG + PDF) to figs/ and print the path."""
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"{name}.{ext}")
    print(f"saved figs/{name}.png (+ .pdf)")

## Why the hardest problem is harder

Class posteriors / features overlap more on harder datasets, so the Bayes-optimal ceiling itself is lower. **Saved: `fig_posterior_overlap`.**

In [ ]:
# WHY the hardest problem is harder: the exact class posteriors p(y|x) overlap more on
# harder datasets, so even the Bayes-optimal classifier errs more. LEFT: the top class
# posterior (max_y p(y|x)) for the easiest vs hardest dataset -- the hard one shifts
# toward chance (1/K), i.e. the classes are less separable. RIGHT: two raw process
# features colored by defect class on the hardest dataset -- the class clouds overlap.
# Reads only the dataset CSVs (data/cluster/*.csv); no model needed.
csvs = {d: root / "data" / "cluster" / f"{d}.csv" for d in manifest["datasets"]}
csvs = {d: p for d, p in csvs.items() if p.exists()}
if csvs:
    def _bayes_err(path):
        dd = pd.read_csv(path, usecols=lambda c: c.startswith("p_"))
        post = dd.to_numpy()
        return float((1 - post.max(1)).mean())
    berr = {d: _bayes_err(p) for d, p in csvs.items()}
    easy, hard = min(berr, key=berr.get), max(berr, key=berr.get)
    de, dh = pd.read_csv(csvs[easy]), pd.read_csv(csvs[hard])
    pc = [c for c in de.columns if c.startswith("p_")]
    cols = de.columns.tolist(); feat = cols[:cols.index(pc[0])]   # raw features precede p_*
    K = len(pc)

    fig, (a0, a1) = plt.subplots(1, 2, figsize=(13, 5))
    for dd, lab, col in [(de, f"easiest: {easy} (Bayes err {berr[easy]:.3f})", COLORS["cascade"]),
                         (dh, f"hardest: {hard} (Bayes err {berr[hard]:.3f})", COLORS["abstention"])]:
        top = dd[pc].to_numpy().max(1)
        a0.hist(top, bins=40, range=(1.0 / K, 1.0), density=True, alpha=0.6, color=col, label=lab)
    a0.axvline(1.0 / K, color="k", ls=":", lw=1, label=f"chance = 1/{K}")
    a0.set_xlabel("top class posterior  max_y p(y|x)"); a0.set_ylabel("density")
    a0.set_title("Class posteriors overlap more on harder data"); a0.legend(fontsize=9)

    sc = dh.sample(min(len(dh), 4000), random_state=0)
    for lab in sorted(sc["defect_label"].unique()):
        m = sc["defect_label"] == lab
        a1.scatter(sc.loc[m, feat[0]], sc.loc[m, feat[1]], s=7, alpha=0.4, label=lab)
    a1.set_xlabel(feat[0]); a1.set_ylabel(feat[1])
    a1.set_title(f"Features overlap by defect class ({hard})")
    a1.legend(fontsize=8, markerscale=2)
    fig.tight_layout(); save_fig(fig, "fig_posterior_overlap"); plt.show()
else:
    print("no dataset CSVs found (need data/cluster/*.csv on the cluster)")

## Risk vs coverage

The selective-classification figure: coverage vs selective accuracy as the payoff `o` sweeps (one operating point per model); star = cascade at full coverage. **Saved: `fig_risk_coverage`.**

In [ ]:
# PAPER FIGURE: risk-coverage curve. Each abstention model (one per payoff o) gives
# one (coverage, selective-accuracy) point at the h=0.5 accept threshold; sweeping
# o = 1 -> 4 traces the frontier. The star marks cascade at full coverage (it never
# abstains). Points up-and-left of a star => abstaining buys accuracy on the boards
# the model chooses to answer.
pay = study("payoff"); core = study("core")
if len(pay):
    sm = seed_mean(pay, ["dataset", "o"], ["coverage@0.5", "selective_acc@0.5"])
    cmap = plt.get_cmap("tab10")
    fig, ax = plt.subplots(figsize=(8.5, 6))
    for j, d in enumerate(sorted(pay["dataset"].unique())):
        s = sm[sm.dataset == d].sort_values("coverage@0.5")
        ax.plot(s["coverage@0.5"], s["selective_acc@0.5"], "-o", ms=4,
                color=cmap(j), label=d)
        cba = core[(core.dataset == d) & (core.loss == "cascade")]["defect_acc"].mean()
        ax.scatter(1.0, cba, marker="*", s=190, color=cmap(j),
                   edgecolor="k", linewidth=0.6, zorder=5)
    ax.set_xlabel("coverage  (fraction of boards the model answers)")
    ax.set_ylabel("selective accuracy  (on the answered boards)")
    ax.set_title("Risk-coverage: abstention trades coverage for accuracy\n"
                 "line = abstention swept over o;   star = cascade at full coverage")
    ax.legend(title="dataset", loc="lower left")
    save_fig(fig, "fig_risk_coverage"); plt.show()
else:
    print("no payoff-study runs with metrics yet")

## Risk (accuracy) vs the payoff o

Forced accuracy + macro-F1 across the `o` sweep. **Saved: `fig_metrics_vs_o`.**

In [ ]:
# The o-sweep on classification quality: FORCED defect accuracy and macro-F1 as the
# payoff o goes 1.0 -> 4.0. macro-F1 climbing toward o=4 (dashed line, where the
# abstention term reduces to cross-entropy for the 3-class head) shows the low-o
# "collapse" is an operating-point choice, not a broken loss.
pay = study("payoff")
if len(pay):
    sm = seed_mean(pay, ["dataset", "o"], ["defect_acc", "defect_macrof1"])
    cmap = plt.get_cmap("tab10")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    for j, d in enumerate(sorted(pay["dataset"].unique())):
        s = sm[sm.dataset == d].sort_values("o")
        axes[0].plot(s["o"], s["defect_acc"], "-o", ms=4, color=cmap(j), label=d)
        axes[1].plot(s["o"], s["defect_macrof1"], "-o", ms=4, color=cmap(j), label=d)
    axes[0].set_title("forced defect accuracy vs payoff o"); axes[0].set_ylabel("defect accuracy")
    axes[1].set_title("macro-F1 (minority-sensitive) vs payoff o"); axes[1].set_ylabel("macro-F1")
    for a in axes:
        a.set_xlabel("payoff o"); a.axvline(4.0, ls="--", color="k", alpha=0.5)
        a.legend(title="dataset")
    save_fig(fig, "fig_metrics_vs_o"); plt.show()
else:
    print("no payoff-study runs with metrics yet")

## Selective-classification analysis (threshold-swept)

Loads the saved checkpoints and sweeps the **rejection threshold** on each model (the standard selective-risk view). Runs on the cluster (needs `results/cluster/*.pt` + `data/cluster/*.csv`); builds `sel_table` for the cells below.

In [ ]:
# Selective-classification analysis (replicates the toy_example). Unlike the cells
# above (which read only the JSON metrics), this loads the SAVED checkpoints and
# sweeps the REJECTION THRESHOLD on each model -- the standard selective-risk view.
# Needs results/cluster/*.pt + data/cluster/*.csv, so run it on the cluster (or after
# rsync-ing them back). Edit SEL_O / SEL_SEED / SEL_COVERAGE to probe other settings.
import sys
sys.path.insert(0, str(root))            # so `import mlp` works regardless of cwd
import torch
import mlp

SEL_O, SEL_SEED, SEL_COVERAGE = 2.0, 0, 0.80
_spec = mlp.load_spec(str(root / "domain" / "smt_paper.yaml"))
_names = mlp.defect_names(_spec)
_ids = mlp.param_ids(_spec)
_covs = np.linspace(0.05, 1.0, 40)


def _find(dataset, loss, o, seed):
    for r in manifest["runs"]:
        if (r["dataset"] == dataset and r["loss"] == loss and r["seed"] == seed
                and (r["o"] == o if loss == "abstention" else True)):
            return r
    return None


def _infer(run, dfd):
    """Test-split predictions for one checkpoint (standardized with its stored stats)."""
    model, ckpt = mlp.load_model(str(root / run["model"]), device="cpu")
    te = np.where(dfd["split"].to_numpy() == "test")[0]
    mu, sd = ckpt["standardize"]["mu"], ckpt["standardize"]["sd"]
    X = ((dfd.iloc[te][_ids].to_numpy(np.float32) - mu) / sd).astype(np.float32)
    y = dfd.iloc[te]["defect_label"].map({n: i for i, n in enumerate(_names)}).to_numpy()
    p = model.predict(torch.from_numpy(X))
    d = {"y": y, "argmax": p["defect_argmax"].cpu().numpy(),
         "real": p["defect_prob"].cpu().numpy(), "te": te,
         "abstain": bool(getattr(model, "abstain", False))}
    if d["abstain"]:
        d["r"] = p["abstain_prob"].cpu().numpy()
    return d


def _risk_cov(score, correct, covs=_covs):
    """Rank-based selective risk: accept the most-confident `cov` fraction (highest
    score first) -> accepted error at each coverage."""
    order = np.argsort(-score); n = len(score)
    return np.array([1 - correct[order[:max(1, int(round(c * n)))]].mean() for c in covs])


def _bayes_err(dfd, te):
    post = dfd.iloc[te][[f"p_{n}" for n in _names]].to_numpy()
    return float((1 - post.max(1)).mean())


sel, _rows = {}, []
for ds in manifest["datasets"]:
    cas, abst = _find(ds, "cascade", None, SEL_SEED), _find(ds, "abstention", SEL_O, SEL_SEED)
    csv = root / "data" / "cluster" / f"{ds}.csv"
    if not (cas and abst and csv.exists()
            and (root / cas["model"]).exists() and (root / abst["model"]).exists()):
        continue
    dfd = pd.read_csv(csv)
    ic, ia = _infer(cas, dfd), _infer(abst, dfd)
    ce_err = 1 - (ic["argmax"] == ic["y"]).mean()
    abst_err = _risk_cov(-ia["r"], (ia["argmax"] == ia["y"]).astype(float))      # reject high r
    conf_err = _risk_cov(ic["real"].max(1), (ic["argmax"] == ic["y"]).astype(float))  # reject low conf
    bayes = _bayes_err(dfd, ia["te"])
    sel[ds] = dict(bayes=bayes, ce_err=float(ce_err), abst_err=abst_err,
                   conf_err=conf_err, ia=ia)
    _at = lambda e: float(e[int(np.argmin(np.abs(_covs - SEL_COVERAGE)))])
    _rows.append(dict(dataset=ds, bayes_err=bayes, ce_full_err=float(ce_err),
                      abst_err_at=_at(abst_err), conf_err_at=_at(conf_err),
                      gain_vs_ce=float(ce_err) - _at(abst_err),
                      gain_vs_conf=_at(conf_err) - _at(abst_err)))

if _rows:
    sel_table = pd.DataFrame(_rows).sort_values("bayes_err").reset_index(drop=True)
    print(f"selective analysis: {len(sel)} datasets  (abstention o={SEL_O:g}, seed {SEL_SEED}, "
          f"reported @ coverage~{SEL_COVERAGE:.2f})")
    print(sel_table.round(4).to_string(index=False))
    print("\ngain_vs_ce   = CE full error - abstention accepted error   (>0: abstaining beats never rejecting)")
    print("gain_vs_conf = cascade confidence error - abstention error   (>0: LEARNED reject beats confidence thresholding)")
else:
    sel_table = pd.DataFrame()
    print("No checkpoints found -- run this on the cluster (needs results/cluster/*.pt + data/cluster/*.csv).")

## Harder tasks: lower accuracy, bigger abstention gain

Two histograms, datasets ordered easy -> hard: accuracy falls, and the gain from abstaining rises. **Saved: `fig_hardness_hist`.**

In [ ]:
# TWO histograms tying difficulty together, datasets ordered easy -> hard by Bayes
# error. LEFT: forced accuracy FALLS as the task gets harder. RIGHT: the accuracy
# GAINED by abstaining RISES as the task gets harder -- so abstention earns its keep
# exactly where the plain classifier struggles. (Needs the selective-compute cell for
# `sel_table`.)
if len(sel_table):
    t = sel_table.sort_values("bayes_err").reset_index(drop=True)
    core = study("core")
    acc = {d: core[(core.dataset == d) & (core.loss == "cascade")]["defect_acc"].mean()
           for d in t["dataset"]}
    order = list(t["dataset"]); x = np.arange(len(order))
    fig, (a0, a1) = plt.subplots(1, 2, figsize=(13, 5))
    a0.bar(x, [acc[d] for d in order], color=COLORS["cascade"])
    a0.set_xticks(x); a0.set_xticklabels(order, rotation=30, ha="right")
    a0.set_ylabel("forced defect accuracy"); a0.grid(axis="x", alpha=0)
    a0.set_ylim(max(0.0, min(acc.values()) - 0.05), 1.0)
    a0.set_title("Accuracy falls as the task gets harder")
    a1.bar(x, t["gain_vs_ce"], color=COLORS["abstention"])
    a1.set_xticks(x); a1.set_xticklabels(order, rotation=30, ha="right")
    a1.axhline(0, color="k", lw=0.8); a1.grid(axis="x", alpha=0)
    a1.set_ylabel(f"accuracy gained by abstaining @ cov~{SEL_COVERAGE:.2f}")
    a1.set_title("Abstention gains more as the task gets harder")
    fig.tight_layout(); save_fig(fig, "fig_hardness_hist"); plt.show()
else:
    print("run the selective-compute cell first (needs sel_table)")

## Comparing the datasets: selective-risk curves

Accepted error vs coverage for every dataset, easy -> hard, vs the confidence baseline / CE / Bayes floor. **Saved: `fig_selective_risk`.**

In [ ]:
# Selective-risk curves (reject threshold swept) per dataset, ordered easy -> hard.
# abstention (reject high reservation) vs cascade confidence thresholding vs the CE
# full-coverage error and the Bayes floor. Abstention dipping below CE as coverage
# drops = it is helping; below the blue line = it beats plain confidence.
if len(sel):
    order = list(sel_table["dataset"])
    ncol = min(5, len(order)); nrow = int(np.ceil(len(order) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 3.0 * nrow), squeeze=False)
    for k, ds in enumerate(order):
        ax = axes[k // ncol][k % ncol]; d = sel[ds]
        ax.plot(_covs, d["abst_err"], "-", color=COLORS["abstention"], lw=2, label="abstention")
        ax.plot(_covs, d["conf_err"], "--", color=COLORS["cascade"], lw=1.8, label="cascade confidence")
        ax.axhline(d["ce_err"], color="#999", ls=":", lw=1.4, label="CE full coverage")
        ax.axhline(d["bayes"], color="k", ls="-", lw=1.0, alpha=0.6, label="Bayes floor")
        ax.set_title(f"{ds} (Bayes {d['bayes']:.3f})", fontsize=10)
        ax.set_xlabel("coverage"); ax.set_ylabel("accepted error"); ax.grid(alpha=0.25)
    for k in range(len(order), nrow * ncol):
        axes[k // ncol][k % ncol].axis("off")
    axes[0][0].legend(fontsize=8)
    fig.tight_layout(); save_fig(fig, "fig_selective_risk"); plt.show()
else:
    print("no checkpoints loaded (see the compute cell above)")

## Optimal configurations

In [ ]:
# Optimal configs. core picks are averaged over seeds so we do not chase noise;
# each study reports the best setting of its own axis per probe dataset.
core = study("core")
mean_over_seeds = core.groupby(["dataset", "loss", "o"], dropna=False).mean(numeric_only=True).reset_index()


def best(metric, maximize=True):
    s = mean_over_seeds.sort_values(metric, ascending=not maximize).iloc[0]
    o = "-" if pd.isna(s["o"]) else f"{s['o']:g}"
    return f"{s['dataset']} / {s['loss']} / o={o}  ->  {metric}={s[metric]:.4f}"


print("=== Optimal core configs (mean over seeds) ===")
print("highest defect accuracy :", best("defect_acc"))
print("smallest gap to Bayes   :", best("gap_to_bayes", maximize=False))
print("best minority macro-F1  :", best("defect_macrof1"))
print("highest joint mechanism :", best("mech_joint"))
print("lowest risk MAE         :", best("risk_mae", maximize=False))

print("\n=== Best setting per study (probe datasets, mean over seeds) ===")
for name, axis, metric, maximize in [
        ("capacity", "hidden", "gap_to_bayes", False),
        ("dropout", "dropout", "gap_to_bayes", False),
        ("lr", "lr", "gap_to_bayes", False),
        ("class_weight", "class_weight", "defect_macrof1", True),
        ("payoff", "o", "selective_acc@0.5", True)]:
    s = study(name)
    if not len(s) or metric not in s.columns or not s[metric].notna().any():
        print(f"  {name:12s}: (no runs yet)"); continue
    sm = seed_mean(s, ["dataset", axis], [metric]).dropna(subset=[metric])
    for ds in sorted(sm["dataset"].unique()):
        win = sm[sm.dataset == ds].sort_values(metric, ascending=not maximize).iloc[0]
        print(f"  {name:12s} [{ds:11s}] best {axis}={win[axis]!s:11s} -> {metric}={win[metric]:.4f}")